# MLPerf Inference — Whisper-large-v3 / LibriSpeech (Speech) on Colab

Runs **whisper-large-v3** on **LibriSpeech dev-clean** (the MLPerf speech2text model +
dataset + WER metric) on a Colab GPU, via `openai-whisper` (the master reference SUT uses
vLLM; openai-whisper gives the same model/data/metric more simply).

Expected WER ≈ **3–4%** on a dev-clean subset.

> Set **Runtime → Change runtime type → GPU** first.

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('cuda',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

name, memory.total [MiB]
Tesla T4, 15360 MiB


cuda True Tesla T4


## 2 · Dependencies (ffmpeg is preinstalled on Colab)

In [ ]:
!pip -q install openai-whisper jiwer soundfile
import whisper, jiwer; print('whisper + jiwer ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 43.3 MB/s eta 0:00:00


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 115.4 MB/s eta 0:00:00


whisper + jiwer ready


## 3 · LibriSpeech dev-clean (free, OpenSLR)

In [ ]:
%%bash
cd /content
if [ ! -d LibriSpeech/dev-clean ]; then wget -q -O d.tgz https://www.openslr.org/resources/12/dev-clean.tar.gz && tar xzf d.tgz && rm d.tgz; fi
echo 'utterances:' $(find LibriSpeech/dev-clean -name '*.flac' | wc -l)

utterances: 2703


## 4 · WER eval (whisper-large-v3, N=30 utterances)

The Blackwell math-SDP guard is harmless on Colab's T4/L4 and keeps the notebook portable.

In [ ]:
%%writefile /content/whisper_eval.py
import os, glob, time, torch
try:
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
except Exception as e: print('SDP guard skipped:', e)
import whisper, jiwer
from whisper.normalizers import EnglishTextNormalizer
N=int(os.environ.get('N_UTT','30')); ROOT='/content/LibriSpeech/dev-clean'
refs={}
for t in glob.glob(os.path.join(ROOT,'*','*','*.trans.txt')):
    for line in open(t):
        u,x=line.strip().split(' ',1); refs[u]=x
items=[]
for f in sorted(glob.glob(os.path.join(ROOT,'*','*','*.flac'))):
    u=os.path.splitext(os.path.basename(f))[0]
    if u in refs: items.append((f,refs[u]))
items=items[:N]; print('utterances:',len(items))
model=whisper.load_model('large-v3', device='cuda' if torch.cuda.is_available() else 'cpu')
norm=EnglishTextNormalizer(); hyps=[]; gts=[]; asec=0.0; t0=time.time()
for f,ref in items:
    a=whisper.load_audio(f); asec+=len(a)/16000.0
    r=model.transcribe(a, language='en', fp16=torch.cuda.is_available(), verbose=False)
    hyps.append(norm(r['text'])); gts.append(norm(ref))
el=time.time()-t0; wer=jiwer.wer(gts,hyps)
print('================================================')
print(f'WER: {wer*100:.2f}%  ({len(items)} utts, LibriSpeech dev-clean)')
print(f'audio {asec:.1f}s | wall {el:.1f}s | RTF {el/asec:.3f}')
print('================================================')

Writing /content/whisper_eval.py


In [ ]:
!cd /content && N_UTT=30 python whisper_eval.py

utterances: 30
  0%|                                              | 0.00/2.88G [00:00<?, ?iB/s]

  2%|▌                                     | 48.1M/2.88G [00:00<00:15, 190MiB/s]

  3%|█▏                                    | 95.0M/2.88G [00:00<00:13, 224MiB/s]

  5%|█▉                                     | 142M/2.88G [00:00<00:12, 236MiB/s]

  7%|██▊                                    | 211M/2.88G [00:01<00:12, 236MiB/s]

...

  0% 0/318 [00:00<?, ?frames/s]

100% 318/318 [00:01<00:00, 268.79frames/s]
  0% 0/204 [00:00<?, ?frames/s]

100% 204/204 [00:01<00:00, 188.35frames/s]
  0% 0/370 [00:00<?, ?frames/s]

100% 370/370 [00:01<00:00, 262.98frames/s]
  0% 0/174 [00:00<?, ?frames/s]

100% 174/174 [00:00<00:00, 182.79frames/s]
WER: 2.16%  (30 utts, LibriSpeech dev-clean)
audio 225.7s | wall 70.4s | RTF 0.312


## Done ✅ — whisper-large-v3 WER on Colab GPU.